# 👁 burst_scan — see it before trusting a number

This notebook computes **no statistics**. It opens the strategy on one name over about eight sessions and walks
its logic in the order it happens, so you can judge by eye whether the code does what the README says:

| § | question | figure |
|---|---|---|
| 1 | which name, which sessions — and where the classifier's probabilities come from | — |
| 2 | **what did the detectors see?** candles, volume against its normal level for that clock time, and the two burst ratios against each side's thresholds | Fig 2 |
| 3 | **what did each sub-strategy decide, and why?** every event of the window: its features, the model's probability, and the reason it traded or did not | table |
| 4 | **what happened after an event?** event cards: entry at the next open, the target and stop lines, the path to the exit | Fig 4 (long), Fig 4 (short) |
| 5 | **did the engine trade what the labels describe?** the engine's own fills drawn over the labelled trades | Fig 5 |
| 6 | **how sensitive is the event set to the thresholds?** the same window under three volume thresholds | Fig 6 |

Times are New York wall clock. ▲ = a long event, ▼ = a short event; a **filled** marker traded, a **hollow** one
was picked by its model but not traded (same-bar clash or its own position still open), a **small grey** one was
below its model's threshold, and a grey × had no model yet (the first training months).

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


VIS_SUBSET = 40  # names the panel (and therefore the classifier) is built on here — the
# probabilities are NOT the full run's; None = all 200 names (~3 minutes)
SYMBOL = None  # None = the name with the most traded events
SESSION = None  # anchor session (a date); None = that name's session with the most trades
N_SESSIONS = 8  # sessions drawn around the anchor
OVERRIDES = {}  # any signals() knob, e.g. {"long_vol_mult": 2.0}
PRM = {**strat.params, **OVERRIDES}  # config.yaml under your overrides — never a bare compose()

## 1 · The name, the sessions and the model behind them

The whole loaded panel is composed once, exactly as `signals()` runs it: the shared context, each side's
detector, the outcome of every event, and the walk-forward classifier that scores each month with a model
trained only on outcomes finished before that month. The name and the anchor session are then picked from the
**out-of-sample** months — by default the name and session with the most traded events, so every figure below
has trades to show.

In [ ]:
panel = rs.load(settings, engine, subset=VIS_SUBSET)
c = compose(panel, **PRM)
ctx = c["ctx"]
tz, bar = panel.tz, panel.bar
ev = pd.concat([t.assign(side=s, why=rs.why(t)) for s, t in c["events"].items()], ignore_index=True)
oos = ev[ev["p"].notna()]
traded_ev = ev[ev["traded"]]
sym = SYMBOL or traded_ev["symbol"].value_counts().index[0]
col = panel.symbols.index(sym)
codes = ctx["codes"]
day_of = pd.Series(ctx["index"].tz_convert(tz).date).groupby(codes).first()
if SESSION is None:
    mine_ev = traded_ev if (traded_ev["symbol"] == sym).any() else oos
    anchor = int(mine_ev.loc[mine_ev["symbol"] == sym, "session"].value_counts().index[0])
else:
    anchor = int(np.flatnonzero(day_of.to_numpy() == pd.Timestamp(SESSION).date())[0])
lo_s = max(anchor - N_SESSIONS + 3, 0)
hi_s = min(lo_s + N_SESSIONS - 1, int(codes.max()))
win = np.flatnonzero((codes >= lo_s) & (codes <= hi_s))
x = ctx["index"][win].tz_convert(tz).tz_localize(None)
print(
    f"panel: {panel.close.shape[0]:,} {bar} bars x {len(panel.symbols)} names; "
    f"{sym}: sessions {day_of[lo_s]} .. {day_of[hi_s]} (anchor {day_of[anchor]})"
)
for side, t in c["events"].items():
    print(
        f"  {side:5s}: {len(t):6,} events, {int(t['p'].notna().sum()):6,} out of sample, "
        f"{int(t['picked'].sum()):5,} picked, {int(t['traded'].sum()):5,} traded"
    )

## 2 · What the detectors saw (Fig 2)

**Top:** the candles, the session VWAP (grey) and every event of both sides in the window. **Middle:** each bar's
volume against the volume this name normally trades **at that clock time** (the median over the previous
`profile_days` sessions — so the routinely busy open is expected, not a burst). **Bottom:** the two burst ratios
over the last `burst_minutes`: volume against its same-clock baseline, and the high–low range against its own.
The dashed lines are each side's thresholds. An event fires on the first bar a side's ratios clear its
thresholds after `cooldown` quiet minutes.

What to look for: events should sit where the bottom ratios jump, not on every busy open; the markers' side and
fill should match the table in §3.

In [ ]:
k = ctx["k"]
expected = bt.volume_profile(panel.volume[[sym]], tz, days=PRM["profile_days"])[sym]
vwap = (panel.close[sym] * panel.volume[sym]).groupby(codes).cumsum() / panel.volume[sym].groupby(
    codes
).cumsum()
here = ev[(ev["symbol"] == sym) & ev["session"].between(lo_s, hi_s)]
breaks = plots._intraday_breaks(pd.DatetimeIndex(x))

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.55, 0.2, 0.25],
    vertical_spacing=0.03,
    subplot_titles=(
        f"{sym} — {bar} candles and events",
        "volume vs its clock-time norm",
        f"burst ratios over {PRM['burst_minutes']} minutes",
    ),
)
fig.add_trace(
    go.Candlestick(
        x=x,
        open=panel.open_[sym].iloc[win],
        high=panel.high[sym].iloc[win],
        low=panel.low[sym].iloc[win],
        close=panel.close[sym].iloc[win],
        name="candles",
        showlegend=False,
        increasing_line_color="#a9c6ba",
        decreasing_line_color="#e0b3b0",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x, y=vwap.iloc[win], name="session VWAP", line={"color": plots._MUTED, "width": 1}
    ),
    row=1,
    col=1,
)
STYLE = {
    "traded": ("", 11),
    "clash: the other side picked it too": ("-open", 11),
    "spaced out: own position still open": ("-open", 11),
    "below the fold's threshold": ("", 6),
    "no model yet (training months)": ("x", 6),
}
for side in SIDES:
    for reason, (suffix, size) in STYLE.items():
        e = here[(here["side"] == side) & (here["why"] == reason)]
        if e.empty:
            continue
        up = side == "long"
        yv = (panel.low[sym] if up else panel.high[sym]).to_numpy()[e["row"]] * (
            0.997 if up else 1.003
        )
        shape = "x" if suffix == "x" else ("triangle-up" if up else "triangle-down") + suffix
        color = COLOR[side] if size > 6 else plots._MUTED
        fig.add_trace(
            go.Scatter(
                x=ctx["index"][e["row"]].tz_convert(tz).tz_localize(None),
                y=yv,
                mode="markers",
                name=f"{side}: {reason}",
                marker={
                    "symbol": shape,
                    "size": size,
                    "color": color,
                    "line": {"color": color, "width": 1.5},
                },
                customdata=np.c_[e["p"], e["thr"], e["reason"]],
                hovertemplate=(
                    "%{x|%a %b %d %H:%M}<br>p %{customdata[0]:.3f} "
                    "(thr %{customdata[1]:.3f})<br>outcome %{customdata[2]} "
                    "(1 target, -1 stop, 0 time)<extra>" + side + "</extra>"
                ),
            ),
            row=1,
            col=1,
        )
fig.add_trace(
    go.Bar(
        x=x, y=panel.volume[sym].iloc[win], name="volume", marker_color="#cfcdc4", showlegend=False
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=expected.iloc[win],
        name="normal at this clock time",
        line={"color": plots.SERIES[4], "width": 1.5},
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=ctx["vol_ratio"][win, col],
        name="volume ratio",
        line={"color": plots.SERIES[4], "width": 1.2},
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=ctx["rng_ratio"][win, col],
        name="range ratio",
        line={"color": plots.SERIES[2], "width": 1.2},
    ),
    row=3,
    col=1,
)
# Thresholds as legend lines (an hline annotation lands at 10^y on a log axis).
for side in SIDES:
    q = S["_side_params"](PRM, side)
    lines = [("volume", q["vol_mult"], "dash")] * (q["trigger"] != "range")
    lines += [("range", q["range_mult"], "dot")] * (q["trigger"] != "volume")
    for what, level, dash in lines:
        fig.add_trace(
            go.Scatter(
                x=[x[0], x[-1]],
                y=[level, level],
                mode="lines",
                name=f"{side} {what} threshold ×{level:g}",
                line={"color": COLOR[side], "dash": dash, "width": 1},
            ),
            row=3,
            col=1,
        )
plots._base(fig, title=f"Fig 2 · {sym}: what the detectors saw", height=860)
below(fig, -0.06)
fig.update_xaxes(rangebreaks=breaks, rangeslider_visible=False)
fig.update_yaxes(type="log", row=3, col=1)
fig.show()

## 3 · What each sub-strategy decided, and why

Every event of the window, both sides. `burst_atr` is the burst's move in session ATRs (its sign is the
direction), `clv` where the burst window closed inside its own range (−1 low … +1 high), `vwap_atr` the distance
from the session VWAP. `p` is the side's out-of-sample probability of reaching its target first, `thr` the fold's
own threshold (the training months' top `trade_q` quantile). `outcome` is what the barrier scan says happened
next — the same thing the engine books (§5).

In [ ]:
table = here.assign(
    time=lambda d: ctx["index"][d["row"]].tz_convert(tz).strftime("%a %m-%d %H:%M"),
    outcome=lambda d: d["reason"].map({1: "target", -1: "stop", 0: "time"}),
    dret_bp=lambda d: 1e4 * d["dret"],
)
table[
    [
        "time",
        "side",
        "log_rvol",
        "log_rrange",
        "burst_atr",
        "clv",
        "vwap_atr",
        "p",
        "thr",
        "why",
        "outcome",
        "dret_bp",
        "nights",
    ]
].round(3).reset_index(drop=True)

## 4 · After the event: event cards (Fig 4)

Six events per side from the whole loaded panel — at least one that reached its **target**, one **stopped**, one
that **timed out**, and one **held overnight** when there is one. Each card runs from ten bars before the event to
three bars after the exit. The shaded strip is the burst window, ● the entry at the next bar's **open**, the green
and red lines the **target** and **stop** — measured from the fill bar's CLOSE, as the engine does
(`stop_entry_price="close"`) and checked from the bar after the fill — and ◆ the exit (green at the target, red at
the stop, grey at the time exit). The x axis counts bars; hover for the clock time.

In [ ]:
def cards(side, n=6, seed=0):
    t = ev[(ev["side"] == side) & ev["complete"]]
    t = t[t["p"].notna()] if t["p"].notna().sum() >= 50 else t  # prefer scored events
    pick = []
    for mask in (t["reason"] == 1, t["reason"] == -1, t["reason"] == 0, t["nights"] > 0):
        cand = t[mask & ~t.index.isin(pick)]
        if len(cand):
            pick.append(cand.sample(1, random_state=seed).index[0])
    rest = t[~t.index.isin(pick) & t["traded"]]
    pick += list(rest.sample(min(n - len(pick), len(rest)), random_state=seed).index)
    return t.loc[pick]


o, h, lo, cl = (f.to_numpy() for f in (panel.open_, panel.high, panel.low, panel.close))
for side in SIDES:
    chosen = cards(side)
    sgn = 1 if side == "long" else -1
    fig = make_subplots(
        rows=2,
        cols=3,
        horizontal_spacing=0.05,
        vertical_spacing=0.14,
        subplot_titles=[
            f"{r.symbol} {ctx['index'][r.row].tz_convert(tz):%m-%d %H:%M} · "
            f"{ {1: 'target', -1: 'stop', 0: 'time'}[r.reason] }"
            f"{' · overnight' if r.nights else ''} · p {'—' if np.isnan(r.p) else f'{r.p:.2f}'}"
            for r in chosen.itertuples()
        ],
    )
    for i, r in enumerate(chosen.itertuples()):
        rr, cc = i // 3 + 1, i % 3 + 1
        a, b = max(r.row - 10, 0), min(int(r.exit_row) + 3, len(o) - 1)
        span = np.arange(a, b + 1)
        stamps = ctx["index"][span].tz_convert(tz).strftime("%a %m-%d %H:%M")
        fig.add_trace(
            go.Candlestick(
                x=span,
                open=o[span, r.col],
                high=h[span, r.col],
                low=lo[span, r.col],
                close=cl[span, r.col],
                text=stamps,
                hoverinfo="text+y",
                showlegend=False,
                increasing_line_color="#a9c6ba",
                decreasing_line_color="#e0b3b0",
            ),
            row=rr,
            col=cc,
        )
        f = r.row + 1
        ref = cl[f, r.col]
        target, stop = ref * (1 + sgn * r.frac), ref * (1 - sgn * r.frac)
        for level, colr, name in ((target, plots._WIN, "target"), (stop, plots._LOSS, "stop")):
            fig.add_trace(
                go.Scatter(
                    x=[f, r.exit_row],
                    y=[level, level],
                    mode="lines",
                    line={"color": colr, "width": 1.5},
                    name=name,
                    showlegend=i == 0,
                ),
                row=rr,
                col=cc,
            )
        fig.add_trace(
            go.Scatter(
                x=[f],
                y=[o[f, r.col]],
                mode="markers",
                name="entry (next open)",
                marker={"symbol": "circle", "size": 9, "color": COLOR[side]},
                showlegend=i == 0,
            ),
            row=rr,
            col=cc,
        )
        ex_color = {1: plots._WIN, -1: plots._LOSS, 0: plots._MUTED}[r.reason]
        fig.add_trace(
            go.Scatter(
                x=[r.exit_row],
                y=[r.exit_px],
                mode="markers",
                name="exit",
                marker={"symbol": "diamond", "size": 10, "color": ex_color},
                showlegend=i == 0,
            ),
            row=rr,
            col=cc,
        )
        fig.add_vrect(
            x0=r.row - k + 0.5,
            x1=r.row + 0.5,
            fillcolor=COLOR[side],
            opacity=0.12,
            line_width=0,
            layer="below",
            row=rr,
            col=cc,
        )  # after the card's traces
    plots._base(fig, title=f"Fig 4 · {side} event cards", height=660)
    below(fig, -0.04)
    fig.update_xaxes(rangeslider_visible=False, showticklabels=False)
    fig.show()
    display(
        chosen[["symbol", "side", *S["FEATURES"], "p", "reason", "dret"]]
        .set_index("symbol")
        .T.round(3)
    )

## 5 · Did the engine trade what the labels describe? (Fig 5)

The panel is backtested with the strategy's own costs and the engine's fills are drawn over the labelled trades
of the window: a labelled entry (hollow circle) should sit under every engine entry (▲/▼) and a labelled exit
(hollow diamond) under every engine exit (×). The reconciliation below it does the same for every trade of the
panel; the only accepted differences are the engine's opposite-entry rule (an opposite-side signal on a held name
closes the position instead of opening its own; an entry on a bar where the shared wallet has no free cash
never fills; a trade still running at the last bar is open at data end).

In [ ]:
fees, slip = bt.strategy_costs(strat, panel, settings)
pf = bt.run(
    panel,
    c["signals"],
    fees=fees,
    slippage=slip,
    init_cash=settings.backtest.init_cash,
    fill=strat.fill,
)
trades = bt.trade_records(pf)
t0, t1 = ctx["index"][win[0]], ctx["index"][win[-1]]
mine = trades[(trades["symbol"] == sym) & (trades["entry_ts"] >= t0) & (trades["entry_ts"] <= t1)]
lab = here[here["traded"]]
fig = go.Figure(
    go.Scatter(
        x=x,
        y=panel.close[sym].iloc[win],
        name=f"{sym} close",
        line={"color": "#bdbab0", "width": 1},
    )
)
for side in SIDES:
    e = lab[lab["side"] == side]
    fig.add_trace(
        go.Scatter(
            x=ctx["index"][e["row"] + 1].tz_convert(tz).tz_localize(None),
            y=e["entry_px"],
            mode="markers",
            name=f"label: {side} entry",
            marker={
                "symbol": "circle-open",
                "size": 14,
                "color": COLOR[side],
                "line": {"color": COLOR[side], "width": 2},
            },
        )
    )
    inside = e[e["exit_row"] <= win[-1]]
    fig.add_trace(
        go.Scatter(
            x=ctx["index"][inside["exit_row"]].tz_convert(tz).tz_localize(None),
            y=inside["exit_px"],
            mode="markers",
            name=f"label: {side} exit",
            marker={
                "symbol": "diamond-open",
                "size": 14,
                "color": COLOR[side],
                "line": {"color": COLOR[side], "width": 2},
            },
        )
    )
for d, sym_ in (("Long", "triangle-up"), ("Short", "triangle-down")):
    tr = mine[mine["direction"] == d]
    fig.add_trace(
        go.Scatter(
            x=tr["entry_ts"].dt.tz_convert(tz).dt.tz_localize(None),
            y=tr["entry_price"],
            mode="markers",
            name=f"engine: {d.lower()} entry",
            marker={"symbol": sym_, "size": 8, "color": plots._INK},
        )
    )
    fig.add_trace(
        go.Scatter(
            x=tr["exit_ts"].dt.tz_convert(tz).dt.tz_localize(None),
            y=tr["exit_price"],
            mode="markers",
            name=f"engine: {d.lower()} exit",
            marker={"symbol": "x", "size": 8, "color": plots._INK},
        )
    )
plots._base(fig, title=f"Fig 5 · {sym}: engine fills over the labelled trades", height=480)
below(fig, -0.12)
fig.update_xaxes(rangebreaks=breaks)
fig.show()
rec = rs.reconcile(panel, c)
display(rec.groupby(["side", "status"]).size().unstack(fill_value=0))

## 6 · The same window under other thresholds (Fig 6)

The events each side's detector would flag in the window if its volume threshold were ×2, ×3 or ×5 (everything
else as configured), drawn on the close. Few markers at ×5 and a carpet at ×2 is the expected picture; the
detector grid of `burst_scan_events.ipynb` §6 is where the thresholds are actually chosen, by a rule fixed in
advance.

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[f"volume ×{v:g}" for v in (2.0, 3.0, 5.0)],
)
counts = {}
for i, vm in enumerate((2.0, 3.0, 5.0), start=1):
    fig.add_trace(
        go.Scatter(
            x=x,
            y=panel.close[sym].iloc[win],
            showlegend=False,
            line={"color": "#bdbab0", "width": 1},
        ),
        row=i,
        col=1,
    )
    for side in SIDES:
        r_, c_ = rs.events(ctx, PRM, side, vol_mult=vm)
        m = (c_ == col) & np.isin(r_, win)
        counts[(vm, side)] = int(len(r_))
        fig.add_trace(
            go.Scatter(
                x=ctx["index"][r_[m]].tz_convert(tz).tz_localize(None),
                y=cl[r_[m], col],
                mode="markers",
                name=side,
                showlegend=i == 1,
                marker={
                    "symbol": "triangle-up" if side == "long" else "triangle-down",
                    "size": 9,
                    "color": COLOR[side],
                },
            ),
            row=i,
            col=1,
        )
plots._base(fig, title=f"Fig 6 · {sym}: events under three volume thresholds", height=640)
fig.update_xaxes(rangebreaks=breaks)
fig.show()
pd.Series(counts, name="events in the whole panel").unstack()

## Checks

Engineering checks only — this notebook judges no edge. A FAIL here means the code does something the README does
not describe; fix it before reading any other notebook.

In [ ]:
mpb = S["_minutes_per_bar"](bar)
first_ok = (ctx["bar_no"][ev["row"]] * mpb >= PRM["skip_open"]).all()
last_ok = (ctx["n_left"][ev["row"]] * mpb >= PRM["skip_close"]).all()
traded = ev[ev["traded"]]
sig = c["signals"]
fill_ok = all(
    sig["entries" if s == "long" else "short_entries"]
    .to_numpy()[
        ev.loc[ev["traded"] & (ev["side"] == s), "row"] + 1,
        ev.loc[ev["traded"] & (ev["side"] == s), "col"],
    ]
    .all()
    for s in SIDES
)
bar_ok = np.allclose(sig["sl_stop"].to_numpy()[traded["row"] + 1, traded["col"]], traded["frac"])
explained = rec["status"].isin(["ok", *rs.ENGINE_RULES]).all()
checks = rs.checks_table(
    [
        ("no event in the first skip_open minutes", bool(first_ok), "all", first_ok),
        ("no event in the last skip_close minutes", bool(last_ok), "all", last_ok),
        ("every traded event enters on the NEXT bar", bool(fill_ok), "all", fill_ok),
        ("barrier frames = each traded event's frac, on the fill bar", bool(bar_ok), "all", bar_ok),
        (
            "engine trades = labels (up to the engine's wallet rules)",
            f"{(rec['status'] == 'ok').mean():.1%} exact",
            "no other status",
            explained,
        ),
    ]
)
display(checks)
rs.scorecard(
    {
        "notebook": "visual",
        "symbol": sym,
        "sessions": [str(day_of[lo_s]), str(day_of[hi_s])],
        "names": len(panel.symbols),
        "events": {s: len(t) for s, t in c["events"].items()},
        "traded": {s: int(t["traded"].sum()) for s, t in c["events"].items()},
        "reconcile": rec["status"].value_counts().to_dict(),
        "checks_failed": int((checks["ok"] == "FAIL").sum()),
    }
)